In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Circuit-Timing visualisieren

<Accordion>
<AccordionItem title="Paketversionen">

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese Versionen oder neuere zu verwenden.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Während die in Qiskit integrierte [Timeline-Ansicht](/guides/visualize-circuit-timing) für statische Circuits nützlich ist, spiegelt sie das Timing von [dynamischen Circuits](/guides/classical-feedforward-and-control-flow) möglicherweise nicht korrekt wider – aufgrund impliziter Operationen wie Broadcasting und Branch-Bestimmung. Als Teil der Unterstützung für dynamische Circuits gibt Qiskit Runtime auf Anfrage genaue Circuit-Timing-Informationen in den Job-Ergebnissen zurück.

> **Note:** - Dies ist eine experimentelle Funktion. Sie befindet sich im Vorschau-Status und kann sich daher ändern.
> - Diese Funktion gilt nur für Qiskit Runtime Sampler-Jobs.
> - Obwohl die gesamte Circuit-Zeit in den Metadaten unter „compilation" zurückgegeben wird, ist dies NICHT die für die Abrechnung verwendete Zeit (QPU-Zeit).
### Timing-Datenabruf aktivieren
Um den Timing-Datenabruf zu aktivieren, setze das experimentelle Flag `scheduler_timing` auf `True`, wenn du den Primitive-Job ausführst.

In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Auf die Circuit-Timing-Daten zugreifen
Wenn angefordert, werden die Circuit-Timing-Daten für jeden PUB in den Job-Ergebnis-Metadaten unter `["compilation"]["scheduler_timing"]["timing"]` zurückgegeben. Dieses Feld enthält die rohen Timing-Informationen. Um die Timing-Informationen anzuzeigen, verwende das integrierte Visualisierungswerkzeug, wie im Abschnitt [Timings visualisieren](#visualize-timings) beschrieben.

Verwende den folgenden Code, um auf die Circuit-Timing-Daten für den ersten PUB zuzugreifen:

In [2]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Die rohen Timing-Daten verstehen
Obwohl das Visualisieren der Circuit-Timing-Daten mit der Methode `draw_circuit_schedule_timing` der häufigste Anwendungsfall ist, kann es nützlich sein, die Struktur der zurückgegebenen rohen Timing-Daten zu verstehen. Das kann dir beispielsweise helfen, Informationen programmatisch zu extrahieren.

Die in `["compilation"]["scheduler_timing"]["timing"]` zurückgegebenen Timing-Daten sind eine Liste von Zeichenketten. Jede Zeichenkette stellt eine einzelne Anweisung auf einem Kanal dar und ist durch Kommas in folgende Datentypen unterteilt:

- `Branch` – Bestimmt, ob die Anweisung in einem Control-Flow-Zweig (then / else) oder im Hauptzweig ist.
- `Instruction` – Das Gate und das Qubit, auf dem es ausgeführt wird.
- `Channel` – Der Kanal, dem die Anweisung zugewiesen ist. Es kann einer der folgenden sein:
      - `Qubit x` – Der Drive-Kanal für Qubit _x_.
      - `AWGRx_y` (arbitrary waveform generator readout) – Wird von Ausleskanälen genutzt, um beim Messen von Qubits zu kommunizieren. Die Argumente _x_ und _y_ entsprechen der Auslesgeräte-ID und der Qubit-Nummer.
- `T0` – Die Startzeit der Anweisung im gesamten Schedule.
- `Duration` – Die Dauer der Anweisung in Einheiten von _dt_ Sekunden, wobei 1 dt = 1 Scheduling-Zyklus. Den `dt`-Wert eines Backends findest du mit [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` – Der Typ der verwendeten Pulsoperation.

Beispiel:

In [3]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### Timings visualisieren
Ab `qiskit-ibm-runtime` v0.43.0 kannst du Circuit-Timings visualisieren. Um die Timings zu visualisieren, musst du zunächst die Ergebnis-Metadaten mit der Methode [`draw_circuit_schedule_timing`](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26) in `fig` umwandeln. Diese Methode gibt eine `plotly`-Abbildung zurück, die du direkt anzeigen, in eine Datei speichern oder beides tun kannst. Weitere Informationen zu den zu verwendenden `plotly`-Befehlen findest du unter [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) und [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html).

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d8287kugbeec73alfbug (DONE)


![Beim Hovern über die Ausgabe werden Informationen wie Start, Ende und Dauer angezeigt.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Beispiel einer generierten Abbildung')
#### Die generierte Abbildung verstehen
Das Bild der von `draw_circuit_schedule_timing` ausgegebenen Circuit-Timing-Daten vermittelt folgende Informationen:

- Die X-Achse ist die Zeit in Einheiten von _dt_ Sekunden, wobei 1 dt = 1 Scheduling-Zyklus. Den `dt`-Wert eines Backends findest du mit [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- Die Y-Achse ist der Kanal (stell dir Kanäle als Instrumente vor, die Pulse ausgeben).
    - `Receive channel` – Der einzige Kanal, der selbst kein Instrument ist. Es ist eine Anweisung, die auf allen Kanälen gespielt wird, die zu diesem Zeitpunkt an einem Kommunikationsverfahren mit dem Hub beteiligt sind.
    - `Qubit x` – Der Drive-Kanal für Qubit x.
    - `AWGRx_y` (arbitrary waveform generator readout) – Wird von Ausleskanälen genutzt, um beim Messen von Qubits zu kommunizieren. Die Argumente _x_ und _y_ entsprechen der Auslesgeräte-ID und der Qubit-Nummer.
    - `Hub` – Steuert das Broadcasting.

Zusätzlich hat jede Anweisung das Format *X_Y*, wobei *X* der Name der Anweisung und *Y* der Pulstyp ist. Ein `play`-Typ wendet Kontrollpulse an, und ein `capture` zeichnet den Zustand des Qubits auf. Du kannst auch über jede Anweisung hovern, um weitere Details zu erhalten. Zum Beispiel zeigt die vorherige Abbildung einen Kontrollpuls für das X-Gate, das bei 1161 dt auf Qubit 10 angewendet wird.
### End-to-End-Beispiel
Dieses Beispiel zeigt dir, wie du die Option aktivierst, die Circuit-Timing-Informationen aus den Metadaten holst und sie als Bild darstellst.

Richte zunächst die Umgebung ein, definiere die Circuits und konvertiere sie zu ISA-Circuits, und definiere und starte die Jobs.

In [5]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,1365,0,shift_phase\nmain,sx_0,Qubit 0,1365,9,play\nmain,sx_0,Qubit 0,1369,0,shift_phase\nmain,rz_0,Qubit 0,1374,0,shift_phase\nmain,barrier,Qubit 0,1374,0,barrier\nmain,measure_0,Qubit 0,1374,64,play\nmain,measure_0,Qubit 0,1438,108,play\nmain,measure_0,AWGR0_0,1485,360,capture\nmain,measure_0,Qubit 0,1546,64,play\nmain,measure_0,Qubit 0,1610,64,play\nmain,barrier,Qubit 0,2046,0,barrier\nmain,broadcast,Hub,1485,561,broadcast\nmain,receive,Receive,2046,7,receive\nthen,x_0,Qubit 0,2061,9,play\nmain,barrier,Qubit 0,2079,0,barrier\nmain,measure_0,Qubit 0,2079,64,play\nmain,measure_0,Qubit 0,2143,108,play\nmain,measure_0,AWGR0_0,2190,360,capture\nmain,measure_0,Qubit 0,2251,64,play\nmain,measure_0,Qubit 0,2315,64,play\nmain,barrier,Qubit 0,2725,0,barrier\nmain,barrier,Qubit 0,2725,0,barrier\n'

Finally, you can visualize and save the timing:

In [6]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Hole anschließend das Circuit-Schedule-Timing: